Two-stage exposure-response analysis of how blackout rate (pct_blackout,
a continuous [0,1] fraction per station-hour) relates to temperature:

  Stage A: Model-free binned exposure-response curve, with confidence
           intervals from a station-clustered bootstrap (resamples whole
           stations, respecting the panel structure -- rows within a
           station are not independent).

  Stage B: Confirmatory fractional-response GLM (quasi-binomial / logit
           link) with temperature entered as a natural cubic spline
           (captures nonlinearity/thresholds instead of assuming a
           straight line), controlling for precipitation, wind speed,
           month, and climate zone, with cluster-robust standard errors
           by station. Includes a likelihood-ratio test of whether the
           nonlinear spline fits significantly better than a linear term.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import patsy
from scipy import stats
from scipy.special import expit
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

INPUT -- edit these paths/column names

In [10]:
INPUT_CSVS = ["/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Energy_resilience/01_data/processed/station_518_forest_perc_50kmBu_district_climate_dummy_df.csv"]
TEMP_COL = "t2m"              # Kelvin in the source data
TEMP_UNITS = "kelvin"         # "kelvin" or "celsius"
OUTCOME_COL = "pct_blackout"  # continuous [0,1] fraction
STATION_COL = "station_id"
CLIMATE_ZONE_COL = "climate_zone_code"
PRECIP_COL = "tp"
WIND_COL = "wind_speed"
DATETIME_COL = "datetime"

N_BOOT = 200          # bootstrap reps for Stage A
SPLINE_DF = 5         # degrees of freedom for the temperature spline
SUBSAMPLE_EVERY_N = 10   # take every Nth row per station for Stage B if full data is too large for memory
PRED_N_SAMPLE = 8000000  # rows used for the final prediction-curve clustered-SE fit

In [4]:
def load_data():
    usecols = [DATETIME_COL, OUTCOME_COL, TEMP_COL, PRECIP_COL, WIND_COL,
               STATION_COL, CLIMATE_ZONE_COL]
    frames = [pd.read_csv(p, parse_dates=[DATETIME_COL], usecols=usecols)
              for p in ["/Users/yiyi/Library/CloudStorage/OneDrive-GeorgiaInstituteofTechnology/Research/Energy_resilience/01_data/processed/station_518_forest_perc_50kmBu_district_climate_dummy_df.csv"]]
    df = pd.concat(frames, ignore_index=True)

    df["t2m_c"] = (df[TEMP_COL] - 273.15 if TEMP_UNITS == "kelvin" else df[TEMP_COL]).astype("float32")
    df[OUTCOME_COL] = df[OUTCOME_COL].astype("float32")
    df[PRECIP_COL] = df[PRECIP_COL].astype("float32")
    df[WIND_COL] = df[WIND_COL].astype("float32")
    df[STATION_COL] = df[STATION_COL].astype("int32")
    df[CLIMATE_ZONE_COL] = df[CLIMATE_ZONE_COL].astype("category")
    df["month"] = df[DATETIME_COL].dt.month.astype("int8")
    return df

# STAGE A: binned exposure-response with station-clustered bootstrap CI

In [3]:
def stage_a_binned(df, bin_width=2, min_n=2000):
    df = df.copy()
    df["t2m_bin"] = ((df["t2m_c"] // bin_width) * bin_width).astype("int16")

    bin_counts = df["t2m_bin"].value_counts()
    valid_bins = bin_counts[bin_counts > min_n].index
    df_b = df[df["t2m_bin"].isin(valid_bins)][[STATION_COL, "t2m_bin", OUTCOME_COL]]

    bins_sorted = sorted(df_b["t2m_bin"].unique())
    grp = df_b.groupby([STATION_COL, "t2m_bin"], observed=True)[OUTCOME_COL].agg(["sum", "count"]).reset_index()
    pivot_sum = grp.pivot(index=STATION_COL, columns="t2m_bin", values="sum").fillna(0).reindex(columns=bins_sorted, fill_value=0)
    pivot_cnt = grp.pivot(index=STATION_COL, columns="t2m_bin", values="count").fillna(0).reindex(columns=bins_sorted, fill_value=0)

    sum_vals, cnt_vals = pivot_sum.values, pivot_cnt.values
    n_stations = sum_vals.shape[0]

    rng = np.random.default_rng(42)
    boot = np.zeros((N_BOOT, len(bins_sorted)))
    for i in range(N_BOOT):
        idx = rng.integers(0, n_stations, size=n_stations)
        boot[i] = sum_vals[idx].sum(axis=0) / cnt_vals[idx].sum(axis=0)

    point = sum_vals.sum(axis=0) / cnt_vals.sum(axis=0)
    result = pd.DataFrame({
        "t2m_bin_c": bins_sorted,
        "mean_pct_blackout": point,
        "ci_lo": np.percentile(boot, 2.5, axis=0),
        "ci_hi": np.percentile(boot, 97.5, axis=0),
        "n_station_hours": cnt_vals.sum(axis=0).astype(int),
    })
    return result

def plot_stage_a(result, out_path="stage_a_curve.png"):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(result["t2m_bin_c"], result["mean_pct_blackout"], color="#c0392b", lw=2, marker="o", ms=4)
    ax.fill_between(result["t2m_bin_c"], result["ci_lo"], result["ci_hi"], color="#c0392b", alpha=0.15,
                     label="95% CI (station-clustered bootstrap)")
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Mean pct_blackout")
    ax.set_title("Stage A: Raw exposure-response curve (unadjusted)")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

# STAGE B: confirmatory spline GLM + likelihood-ratio test + predicted curve

In [5]:
def subsample_per_station(df, every_n):
    df = df.sort_values(STATION_COL).reset_index(drop=True)
    row_in_group = df.groupby(STATION_COL, observed=True).cumcount()
    return df[row_in_group % every_n == 0].reset_index(drop=True)


def stage_b_lr_test(df_sample):
    """Likelihood-ratio test: does the nonlinear spline fit better than a linear temperature term?"""
    f_spline = (f"{OUTCOME_COL} ~ cr(t2m_c, df={SPLINE_DF}, constraints='center') "
                f"+ {PRECIP_COL} + {WIND_COL} + C(month) + C({CLIMATE_ZONE_COL})")
    f_linear = (f"{OUTCOME_COL} ~ t2m_c + {PRECIP_COL} + {WIND_COL} + C(month) + C({CLIMATE_ZONE_COL})")

    m_spline = sm.formula.glm(f_spline, df_sample, family=sm.families.Binomial()).fit()
    m_linear = sm.formula.glm(f_linear, df_sample, family=sm.families.Binomial()).fit()

    lr_stat = 2 * (m_spline.llf - m_linear.llf)
    df_diff = m_spline.df_model - m_linear.df_model
    p_value = stats.chi2.sf(lr_stat, df_diff)
    return lr_stat, df_diff, p_value, f_spline


def stage_b_predict_curve(df_sample, formula_rhs, n_sample=PRED_N_SAMPLE, random_state=1):
    """Fit with cluster-robust SEs on a tractable subsample, then predict an
    adjusted exposure-response curve (holding other covariates at representative values)."""
    df_fit = df_sample.sample(n=min(n_sample, len(df_sample)), random_state=random_state)

    rhs = formula_rhs.split("~", 1)[1] if "~" in formula_rhs else formula_rhs
    y, X = patsy.dmatrices(f"{OUTCOME_COL} ~ {rhs}", df_fit, return_type="dataframe")
    model = sm.GLM(y, X, family=sm.families.Binomial())
    res = model.fit(cov_type="cluster", cov_kwds={"groups": df_fit[STATION_COL]})

    t_grid = np.arange(df_sample["t2m_c"].quantile(0.01), df_sample["t2m_c"].quantile(0.99), 0.5)
    modal_zone = df_sample[CLIMATE_ZONE_COL].mode().iloc[0]
    pred_df = pd.DataFrame({
        "t2m_c": t_grid,
        PRECIP_COL: df_sample[PRECIP_COL].median(),
        WIND_COL: df_sample[WIND_COL].median(),
        "month": 6,
        CLIMATE_ZONE_COL: pd.Categorical([modal_zone] * len(t_grid), categories=df_sample[CLIMATE_ZONE_COL].cat.categories),
    })
    Xp = np.asarray(patsy.build_design_matrices([X.design_info], pred_df)[0])

    eta = Xp @ res.params.values
    se_eta = np.sqrt(np.einsum("ij,jk,ik->i", Xp, res.cov_params().values, Xp))
    pred, lo, hi = expit(eta), expit(eta - 1.96 * se_eta), expit(eta + 1.96 * se_eta)

    out = pd.DataFrame({"t2m_c": t_grid, "pred": pred, "lo": lo, "hi": hi})

    deriv = np.gradient(pred, t_grid)
    mask = (t_grid > 15) & (t_grid < 40)
    steepest_t = float(t_grid[mask][np.argmax(deriv[mask])]) if mask.any() else None
    return out, res, steepest_t


def plot_stage_b(out, steepest_t, out_path="stage_b_curve.png"):
    fig, ax = plt.subplots(figsize=(9, 5.5))
    ax.plot(out["t2m_c"], out["pred"], color="#2c3e50", lw=2)
    ax.fill_between(out["t2m_c"], out["lo"], out["hi"], color="#2c3e50", alpha=0.15, label="95% CI (cluster-robust)")
    if steepest_t is not None:
        ax.axvline(steepest_t, color="#c0392b", ls="--", lw=1, label=f"Steepest slope ≈ {steepest_t:.1f}°C")
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Adjusted predicted pct_blackout")
    ax.set_title("Stage B: Model-adjusted exposure-response curve\n(controlling for precip, wind, month, climate zone)")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

# STAGE B+: curve with sample-size panel (sanity check for wide tails)

In [ ]:
def plot_stage_b_with_counts(pred_curve, stage_a_result, steepest_t, out_path="stage_b_with_counts.png"):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7.5), sharex=True,
                                     gridspec_kw={"height_ratios": [3, 1]})
    ax1.plot(pred_curve["t2m_c"], pred_curve["pred"], color="#2c3e50", lw=2)
    ax1.fill_between(pred_curve["t2m_c"], pred_curve["lo"], pred_curve["hi"], color="#2c3e50", alpha=0.15,
                      label="95% CI (cluster-robust)")
    if steepest_t is not None:
        ax1.axvline(steepest_t, color="#c0392b", ls="--", lw=1, label=f"Steepest slope ≈ {steepest_t:.1f}°C")
    ax1.set_ylabel("Adjusted predicted pct_blackout")
    ax1.set_title("Model-adjusted exposure-response curve, with sample size")
    ax1.legend()
    ax1.grid(alpha=0.3)
 
    ax2.bar(stage_a_result["t2m_bin_c"], stage_a_result["n_station_hours"], width=1.8, color="#7f8c8d")
    ax2.set_yscale("log")
    ax2.set_xlabel("Temperature (°C)")
    ax2.set_ylabel("Station-hours\n(log scale)")
    ax2.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

,datetime,pct_blackout,t2m,tp,wind_speed,station_id,climate_zone_code,t2m_c,month
0,2014-11-17 17:00:00,0.000000,295.09723,5.645638e-04,1.310930,495,Aw,21.947229,11
1,2014-11-29 17:00:00,0.000000,293.04870,0.000000e+00,2.215274,495,Aw,19.898701,11
2,2014-11-30 22:00:00,0.000000,289.83447,4.351137e-07,2.000125,495,Aw,16.684469,11
3,2014-11-13 07:00:00,0.000000,297.14110,6.666072e-04,3.900476,495,Aw,23.991100,11
4,2014-12-31 15:00:00,0.000000,296.46838,2.732359e-06,2.988734,495,Aw,23.318380,12
...,...,...,...,...,...,...,...,...,...
7044374,2016-06-03 15:00:00,0.000000,310.37060,5.161390e-06,1.261605,24,BSh,37.220600,6
7044375,2016-11-23 13:00:00,0.133333,298.23236,0.000000e+00,1.544500,24,BSh,25.082359,11
7044376,2016-12-01 01:00:00,0.000000,291.30194,0.000000e+00,0.753274,24,BSh,18.151939,12
7044377,2016-05-21 07:00:00,0.000000,312.40796,0.000000e+00,6.349144,24,BSh,39.257961,5


# STAGE C: climate-zone-stratified curves (does the threshold shift by zone?)

In [ ]:
def stage_c_by_zone(df, spline_df=4, every_n=1, fit_n_sample=500_000, min_rows=5000, min_stations=2):
    """Fit a separate spline GLM per climate zone and return each zone's
    predicted curve + its steepest-slope ('threshold') temperature.
    Zones with few stations will have noisier/overfit curves -- check
    station counts before trusting any wiggle in the curve shape."""
    results = {}
    for zone in df[CLIMATE_ZONE_COL].cat.categories:
        sub = df[df[CLIMATE_ZONE_COL] == zone].copy()
        n_stations = sub[STATION_COL].nunique()
        if len(sub) < min_rows or n_stations < min_stations:
            print(f"  {zone}: skipped (n={len(sub)}, stations={n_stations})")
            continue
 
        sub = sub.sort_values(STATION_COL)
        row_in_group = sub.groupby(STATION_COL, observed=True).cumcount()
        sub_s = sub[row_in_group % every_n == 0].reset_index(drop=True)
        fit_sample = sub_s.sample(n=min(len(sub_s), fit_n_sample), random_state=1)
 
        formula_rhs = f"cr(t2m_c, df={spline_df}, constraints='center') + {PRECIP_COL} + {WIND_COL} + C(month)"
        y, X = patsy.dmatrices(f"{OUTCOME_COL} ~ {formula_rhs}", fit_sample, return_type="dataframe")
        res = sm.GLM(y, X, family=sm.families.Binomial()).fit(
            cov_type="cluster", cov_kwds={"groups": fit_sample[STATION_COL]})
 
        t_grid = np.arange(sub_s["t2m_c"].quantile(0.02), sub_s["t2m_c"].quantile(0.98), 0.5)
        pred_df = pd.DataFrame({
            "t2m_c": t_grid,
            PRECIP_COL: sub_s[PRECIP_COL].median(),
            WIND_COL: sub_s[WIND_COL].median(),
            "month": 6,
        })
        Xp = np.asarray(patsy.build_design_matrices([X.design_info], pred_df)[0])
        eta = Xp @ res.params.values
        se = np.sqrt(np.einsum("ij,jk,ik->i", Xp, res.cov_params().values, Xp))
        pred = expit(eta)
        steepest_t = float(t_grid[np.argmax(np.gradient(pred, t_grid))])
 
        results[zone] = pd.DataFrame({
            "t2m_c": t_grid, "pred": pred,
            "lo": expit(eta - 1.96 * se), "hi": expit(eta + 1.96 * se),
        })
        print(f"  {zone}: n={len(sub):,} ({n_stations} stations), steepest slope ~{steepest_t:.1f}C")
    return results
 
 
def plot_stage_c(results, out_path="stage_c_by_zone.png"):
    fig, ax = plt.subplots(figsize=(9.5, 6))
    for zone, r in results.items():
        line, = ax.plot(r["t2m_c"], r["pred"], lw=2, label=zone)
        ax.fill_between(r["t2m_c"], r["lo"], r["hi"], color=line.get_color(), alpha=0.1)
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Adjusted predicted pct_blackout")
    ax.set_title("Exposure-response curve by climate zone\n(controlling for precip, wind, month)")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

In [11]:
print("Loading data...")
df = load_data()
print(f"  {len(df):,} rows, {df[STATION_COL].nunique()} stations")

print("\n--- Stage A: binned exposure-response ---")
stage_a_result = stage_a_binned(df)
stage_a_result.to_csv("stage_a_binned_result.csv", index=False)
plot_stage_a(stage_a_result)
print(stage_a_result.to_string(index=False))

print("\n--- Stage B: confirmatory spline GLM ---")
df_sample = df # run on full dataset
print(f"  Using {len(df_sample):,} rows (every {SUBSAMPLE_EVERY_N}th per station) for fitting")

lr_stat, df_diff, p_value, formula = stage_b_lr_test(df_sample)
print(f"  LR test (nonlinear vs. linear): stat={lr_stat:.2f}, df={df_diff:.0f}, p={p_value:.3g}")

pred_curve, res, steepest_t = stage_b_predict_curve(df_sample, formula)
pred_curve.to_csv("stage_b_predicted_curve.csv", index=False)
plot_stage_b(pred_curve, steepest_t)
print(f"  Steepest-slope (approx. threshold) temperature: {steepest_t:.1f}°C" if steepest_t else "  (no clear steepest point found)")

print("\n--- Stage B+: curve with sample-size panel ---")
plot_stage_b_with_counts(pred_curve, stage_a_result, steepest_t)

print("\n--- Stage C: climate-zone-stratified thresholds ---")
zone_results = stage_c_by_zone(df)
plot_stage_c(zone_results)

print("\nDone. Outputs written: stage_a_binned_result.csv, stage_a_curve.png, "
        "stage_b_predicted_curve.csv, stage_b_curve.png, stage_b_with_counts.png, "
        "stage_c_by_zone.png")

Loading data...
  7,044,379 rows, 516 stations

--- Stage A: binned exposure-response ---
 t2m_bin_c  mean_pct_blackout    ci_lo    ci_hi  n_station_hours
         0           0.042206 0.025952 0.069704             2799
         2           0.055557 0.036273 0.074560             4634
         4           0.040066 0.026637 0.056008            10842
         6           0.053610 0.040969 0.067603            26863
         8           0.085421 0.073033 0.101012            55743
        10           0.108773 0.094164 0.129181            97750
        12           0.113715 0.096292 0.130979           146400
        14           0.100744 0.084960 0.116939           213186
        16           0.091618 0.077864 0.105891           301849
        18           0.089014 0.076705 0.102153           412478
        20           0.078156 0.067340 0.088765           666407
        22           0.083823 0.070360 0.096391           873755
        24           0.098610 0.085022 0.111164          1042410


NameError: name 'plot_stage_b_with_counts' is not defined